# AAL Sales Analysis · Q4 2020

**Applied Data Science with Python | Course-end project**  
**Working style:** follow the assignment's four sections in order. This is the complete, executable report for all four assignment sections.

| Rubric section | Status |
| --- | --- |
| **1. Data Wrangling** (a–d) | Complete |
| **2. Data Analysis** (a–d, plus optional extensions) | Complete |
| **3. Data Visualization** | Complete |
| **4. Report Generation** | Complete |

The data describes apparel sales in Australia. Its source does **not** specify a currency, so we refer to the `Sales` column as recorded sales values without assuming AUD.

**Before running:** place the supplied `AusApparalSales4thQrt2020.csv` beside this notebook in Practice Labs. If working from the downloaded project folder, leave the original ZIP in the `data/` directory. Both arrangements are supported.

## 1. Data Wrangling

**Rubric goal:** (a) check missing/incorrect records, (b) justify how to handle them, (c) apply normalization, and (d) demonstrate and explain `GroupBy()`.

### 1a. Inspect missing and incorrect entries

**Step 1: Load the original dataset.** `pandas` (usually imported as `pd`) reads the CSV into a *DataFrame*: a table with rows and columns. This cell also accepts the original ZIP if you are working from the project folder.

In [ ]:
from pathlib import Path
import pandas as pd

# The first existing file wins. Practice Labs usually keeps CSV and notebook together.
candidates = [
    Path('AusApparalSales4thQrt2020.csv'),
    Path('data/AusApparalSales4thQrt2020.csv'),
    Path('../data/AusApparalSales4thQrt2020.csv'),
    Path('1739523823_ausapparalsales4thqrt2020.zip'),
    Path('data/1739523823_ausapparalsales4thqrt2020.zip'),
    Path('../data/1739523823_ausapparalsales4thqrt2020.zip'),
]
source_file = next((path for path in candidates if path.is_file()), None)
if source_file is None:
    raise FileNotFoundError('Put the CSV beside this notebook, or the original ZIP in data/.')

raw_df = pd.read_csv(source_file)
print('Source file:', source_file)
print(f'Rows: {len(raw_df):,}; columns: {len(raw_df.columns)}')
display(raw_df.head())

**Step 2: Check data types and missing cells.** `isna()` gives `True` for missing values; `notna()` does the opposite. Summing Boolean values counts them.

In [ ]:
print('Original data types:')
display(raw_df.dtypes.to_frame('dtype'))

missing_check = pd.DataFrame({
    'missing: isna()': raw_df.isna().sum(),
    'present: notna()': raw_df.notna().sum(),
})
display(missing_check)

**Step 3: Clean a copy, not the source.** The text fields have leading spaces, so we remove them with `.str.strip()`. We convert dates and numeric fields explicitly. `errors='coerce'` turns an invalid entry into a detectable missing value rather than hiding a parsing error.

In [ ]:
df = raw_df.copy()  # The original raw_df stays unchanged.
df.columns = df.columns.str.strip()

for column in ['Time', 'State', 'Group']:
    df[column] = df[column].astype('string').str.strip()

df['Date'] = pd.to_datetime(
    df['Date'].astype('string').str.strip(), format='%d-%b-%Y', errors='coerce'
)
for column in ['Unit', 'Sales']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Example of removing extra spaces:')
print('Before:', repr(raw_df.loc[0, 'Group']))
print('After: ', repr(df.loc[0, 'Group']))

display(df.head())

**Step 4: Inspect potential errors after cleaning.** A *duplicate business key* would mean more than one entry for the same Date × Time × State × Group. Negative sales might be returns in another business, so we flag them for investigation rather than deleting them automatically.

In [ ]:
print('Missing or invalid values after conversion:')
display(df.isna().sum().to_frame('Count'))

check_keys = ['Date', 'Time', 'State', 'Group']
quarter_start = pd.Timestamp('2020-10-01')
quarter_end = pd.Timestamp('2020-12-31')

issues = pd.Series({
    'Identical duplicate rows': int(df.duplicated().sum()),
    'Duplicate date/time/state/group keys': int(df.duplicated(check_keys).sum()),
    'Out-of-quarter dates': int((df['Date'].notna() & ~df['Date'].between(quarter_start, quarter_end)).sum()),
    'Units <= 0': int((df['Unit'] <= 0).sum()),
    'Fractional units': int(((df['Unit'] % 1 != 0) & df['Unit'].notna()).sum()),
    'Negative sales values': int((df['Sales'] < 0).sum()),
    'Unexpected group labels': int((df['Group'].notna() & ~df['Group'].isin(['Kids', 'Men', 'Women', 'Seniors'])).sum()),
    'Unexpected time labels': int((df['Time'].notna() & ~df['Time'].isin(['Morning', 'Afternoon', 'Evening'])).sum()),
})
display(issues.to_frame('Rows flagged'))

for column in ['State', 'Group', 'Time']:
    print(column, ':', sorted(df[column].dropna().unique().tolist()))

**Step 5: Check whether any *whole days* are absent.** A dataset can have zero `NaN` cells while missing entire days of trading. Q4 2020 has **92 calendar days**, so we compare that calendar with the distinct dates in the file.

In [ ]:
quarter_dates = pd.date_range('2020-10-01', '2020-12-31', freq='D')
observed_dates = pd.DatetimeIndex(df['Date'].dropna().unique())
missing_dates = quarter_dates.difference(observed_dates)

print('Recorded calendar days:', len(observed_dates), '/', len(quarter_dates))
print('Unrecorded dates:')
for day in missing_dates:
    print(' ', day.strftime('%d %b %Y'))

rows_each_day = df.groupby('Date').size()
print('Rows per recorded day: minimum', rows_each_day.min(), 'maximum', rows_each_day.max())

### 1b. Recommend how to treat missing or incorrect information

**Decision for this dataset:**

| Finding | Recommended treatment | Reason |
| --- | --- | --- |
| Missing values in existing rows | **No filling or dropping required** if the checks above show zero | We should not change valid data without a reason. |
| Extra spaces in text fields | **Strip whitespace** | Otherwise `' Men'` and `'Men'` could be treated as different categories. |
| Incorrect dates, amounts or duplicate keys (if found later) | **Investigate against the source before dropping or filling** | Automatic fixes can create false business results. |
| Entire dates absent (31 October and 31 December 2020) | **Keep them identified as unrecorded; do not fill with zero or invented averages** | A missing day does not prove the business earned no sales. |

**Hypothesis, unverified:** 31 October is Halloween and 31 December is New Year's Eve. Holiday-related closures or altered trading *could* explain the two gaps, but a reporting or extraction omission is also possible. The file contains no opening-hours data. Halloween is not generally a day when Australian retailers close, so neither date should be labelled a confirmed closure without evidence.

**Why not `dropna()` or `fillna()` here?** Those functions operate on *existing rows*, whereas two entire days are missing from the file. We would need independent trading schedules or original transaction records to resolve that question.

### 1c. Apply Min-Max normalization

**Chosen method: Min-Max normalization**, as recommended by the assignment. It rescales each numeric column to a value between **0 and 1**:

\[
 x_{\mathrm{normalized}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
\]

We will normalize **`Unit`** and **`Sales`**, leaving the originals intact. We should **not** normalize dates or category labels.

In [ ]:
for column in ['Unit', 'Sales']:
    minimum = df[column].min()
    maximum = df[column].max()

    if maximum == minimum:
        raise ValueError(f'Cannot apply Min-Max to {column}: all values are identical.')

    df[f'{column}_normalized'] = (df[column] - minimum) / (maximum - minimum)

print('Original and normalized values:')
display(df[['Unit', 'Unit_normalized', 'Sales', 'Sales_normalized']].head(8))

print('Normalized minimum and maximum (both must be 0 and 1):')
display(df[['Unit_normalized', 'Sales_normalized']].agg(['min', 'max']))

**Why might the two normalized columns be identical?** Inspect the ratio `Sales / Unit`. In this particular dataset, each sale value equals **2,500 × units**, so both columns carry the same relative information. Normalization does **not** make them independent.

In [ ]:
ratio_values = sorted((df['Sales'] / df['Unit']).dropna().unique().tolist())
print('Distinct Sales / Unit ratios:', ratio_values)
print('Greatest difference between normalized columns:',
      (df['Sales_normalized'] - df['Unit_normalized']).abs().max())

### 1d. Apply `GroupBy()` and recommend its use

`groupby()` uses **split → apply → combine**: split rows into logical groups, calculate a value per group, then combine the results into one summary table.

For AAL, **grouping (data chunking)** is more useful than merging at this stage. We already have all columns in one table, and we need state-wise and demographic summaries. A **merge** would be appropriate later if we obtained a *different table*, such as store locations or marketing spend.

**Important:** aggregate the original `Sales` and `Unit` values, not the normalized columns. Sums of normalized scores are not meaningful revenue totals.

In [ ]:
by_state = (
    df.groupby('State', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
)

by_state_and_group = (
    df.groupby(['State', 'Group'], as_index=False)
      .agg(Records=('Sales', 'size'), Total_Sales=('Sales', 'sum'))
      .sort_values(['State', 'Group'])
)

print('State-level groupings:')
display(by_state)
print('First ten state-by-group combinations:')
display(by_state_and_group.head(10))

**Section 1 verification.** The following quick checks are meant to catch mistakes before we begin the statistical analysis. They do not change any data.

In [ ]:
assert len(raw_df) == len(df), 'Rows must not disappear during cleaning.'
assert df[['Unit_normalized', 'Sales_normalized']].notna().all().all()
assert df['Unit_normalized'].between(0, 1).all()
assert df['Sales_normalized'].between(0, 1).all()
assert abs(by_state['Total_Sales'].sum() - df['Sales'].sum()) < 0.000001
assert (df['Unit_normalized'] - df['Sales_normalized']).abs().max() < 0.000000001
print('Section 1 checks passed: row count preserved, normalized values in [0,1], grouped sales reconciled.')

**Section 1 recap.** We checked data quality, documented the missing-date treatment, normalized without changing source values, and practised `GroupBy()`. Now we can analyse the *original* Sales and Unit columns.

---
## 2. Data Analysis

**Rubric goal:** (a) descriptive statistics for Unit and Sales, (b) identify highest/lowest selling demographic groups, (c) compare the highest/lowest selling states, and (d) prepare weekly, monthly and quarterly reports.

The original assignment repeats the group question in 2b and 2c. We use 2c to answer the wider project question about states, without ignoring the duplicate wording.

### 2a. Calculate descriptive statistics

We will find **mean**, **median**, **mode** and **standard deviation** for both numeric columns:

- **Mean:** arithmetic average (sum divided by number of rows).
- **Median:** the middle value when data is ordered.
- **Mode:** the value appearing most often.
- **Standard deviation:** how much values vary around the mean. Pandas `std()` uses the *sample* standard deviation (`ddof=1`).

Run this cell **after Section 1**, because it uses our cleaned `df`.

In [ ]:
# Calculate the summary for each original numeric column.
statistics = df[['Unit', 'Sales']].agg(['mean', 'median', 'std']).T
statistics['mode'] = [df[column].mode().iloc[0] for column in ['Unit', 'Sales']]
statistics = statistics[['mean', 'median', 'mode', 'std']]

print('Descriptive statistics for each recorded row:')
display(statistics.round(2))
print(f"Sales mode: {statistics.loc['Sales', 'mode']:,.0f}")

**What did we learn?** The recorded mean sale is about **45,013.56**, the median is **35,000**, and the most common sale value is **22,500** (currency unspecified). The mean being above the median suggests larger transactions pull the average upward. These are values **per dataset row**, not per individual shop or customer; the dataset does not identify transactions.

### 2b. Find the highest and lowest selling demographic groups

We will group the original **Sales** and **Unit** fields by `Group`. We use the *total* sales value to rank Kids, Men, Women and Seniors. Counts are included so we can see whether groups have similar numbers of records.

In [ ]:
group_report = (
    df.groupby('Group', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)

display(group_report)
print('Highest selling group:', group_report.iloc[0]['Group'])
print('Lowest selling group: ', group_report.iloc[-1]['Group'])

**Observation:** Men have the highest recorded total (**85,750,000**), and Seniors the lowest (**84,037,500**). Each demographic group contributes **1,890** records. The gap is modest compared with the overall size of the dataset; these totals alone do not explain *why* a group leads.

### 2c. Find the highest and lowest selling states or territories

The assignment's 2b and 2c repeat the group question, but the project statement explicitly asks for **state-by-state sales**. We address this here, ranking states by the sum of original Sales.

In [ ]:
state_report = (
    df.groupby('State', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)

display(state_report)
print('Highest selling state:', state_report.iloc[0]['State'])
print('Lowest selling state: ', state_report.iloc[-1]['State'])

**Observation:** Victoria (VIC) is highest at **105,565,000** and Western Australia (WA) is lowest at **22,152,500**. Every recorded state/territory has **1,080** rows. We cannot infer store productivity or market demand without store counts, populations, or other context. The source does not include ACT.

**Investigate the lowest-selling states by demographic (part of 2c).**

Before making a marketing recommendation, we should ask: **within each lower-selling state, which customer group contributes the most and least?** We use `groupby(['State', 'Group'])` to build a state-by-demographic sales table.

These are *recorded sales totals*, not profit. Each state has the same number of rows for each demographic, which makes within-state comparisons straightforward. Comparing absolute state totals is more difficult because we do not know store counts or local market sizes.

In [ ]:
# Compare all four demographic groups within each state.
segment_sales = df.groupby(['State', 'Group'])['Sales'].sum().unstack('Group')
segment_share = 100 * segment_sales.div(segment_sales.sum(axis=1), axis=0)

# Choose the four states or territories with the lowest TOTAL recorded sales.
low_state_names = state_report.nsmallest(4, 'Total_Sales')['State']

low_state_demographics = pd.DataFrame({
    'Total_Sales': segment_sales.sum(axis=1),
    'Strongest_Group': segment_sales.idxmax(axis=1),
    'Strongest_Sales': segment_sales.max(axis=1),
    'Lowest_Group': segment_sales.idxmin(axis=1),
    'Lowest_Sales': segment_sales.min(axis=1),
    'Lowest_Share_pct': segment_share.min(axis=1).round(2),
    'Group_Gap_pp': (segment_share.max(axis=1) - segment_share.min(axis=1)).round(2),
}).loc[low_state_names].reset_index()

print('Sales by demographic in the four lowest-selling states:')
display(segment_sales.loc[low_state_names])
print('Summary: strongest, lowest, and the lowest group share in each state:')
display(low_state_demographics)
print('All-state within-state group share range:',
      f'{segment_share.min().min():.2f}% to {segment_share.max().max():.2f}%')

**What the breakdown suggests (hypotheses, not conclusions):**

| Lower-selling area | Strongest group | Lowest group | Sensible *test*, not a cut |
|---|---|---|---|
| WA | Men | Women | Test whether a Women's offer, assortment change or marketing message improves sales |
| NT | Men | Seniors | Investigate Seniors' needs and test a targeted offer |
| TAS | Kids | Women | Test Women's assortment or a family-linked campaign |
| QLD | Kids | Seniors | Investigate Seniors' demand while testing family-oriented promotions |

The key finding is **relative balance**: all four groups contribute approximately one quarter of each state's recorded sales. The weakest observed share is **23.76% (Women in WA)**, and even the largest within-state difference (WA) is only **2.21 percentage points**. The data does **not** support removing any demographic group.

**Why not drop a group immediately?** Revenue is not profit. A decision to reduce or remove a range needs evidence on **gross margins, inventory costs, stock turnover, returns, store counts, customer footfall, local demand**, and any equality or customer-access consequences. A controlled promotional or assortment test is less risky. If a range is consistently unprofitable *after* those checks, consider a limited product-space adjustment before removing an entire demographic offering.

We'll use these findings when writing Section 4's recommendations; Section 3 will handle the charts.

### 2d. Produce weekly, monthly and quarterly reports

A **time-based GroupBy** creates reports at different levels of detail. We first convert each date to a weekly or monthly **period**; this labels which calendar bucket each row belongs to. We sum original sales and units and count distinct **recorded** days. No missing days are filled or estimated.

**Weekly:** We define weeks as **Monday to Sunday**, using `W-SUN`. The first and last weeks extend beyond Q4, so their totals only include Q4 records from the source; those rows are not comparable with complete weeks without adjusting for day counts.

In [ ]:
weekly_report = (
    df.assign(Week=df['Date'].dt.to_period('W-SUN'))
      .groupby('Week', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Week')
)

print('Weekly sales report (Monday-Sunday calendar weeks):')
display(weekly_report)
print('Number of weekly reporting periods:', len(weekly_report))

**Monthly:** Converting to a monthly period with `M` produces one total for **October, November and December**. `Recorded_Days` shows why two months have incomplete calendar coverage: October and December each have only 30 recorded days, and November has 30 too because it has 30 calendar days.

In [ ]:
monthly_report = (
    df.assign(Month=df['Date'].dt.to_period('M'))
      .groupby('Month', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Month')
)

print('Monthly sales report:')
display(monthly_report)

**Quarterly:** The `Q` period groups the entire quarter together. The quarter contains **90 recorded dates**, although October through December have **92 calendar dates**. We report recorded values only and flag the two missing dates as unverified gaps.

In [ ]:
quarterly_report = (
    df.assign(Quarter=df['Date'].dt.to_period('Q'))
      .groupby('Quarter', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Quarter')
)

print('Quarterly sales report:')
display(quarterly_report)
print('Quarter total recorded sales:', f"{quarterly_report['Total_Sales'].sum():,.0f}")
print('Quarter labels:', quarterly_report['Quarter'].astype(str).tolist())

### 2e. Extra business question: Should we focus on higher or lower spend?

**Important distinction: we can measure higher- or lower-*volume recorded periods*, not individual customer spending.** Each row describes a date, state, demographic and time of day, aggregating the number of units and sales. It is **not an individual customer** or basket. The file contains no transaction count, customer ID, item price variation, cost or profit, so we cannot calculate average basket value or determine whether customers spent more per visit.

Our first useful comparison is **average units per recorded period** for each state and group. Because all state/group combinations have the same 270 observed periods, this is comparable *within each state*. Because `Sales = Unit × 2,500` on every row, sales per record tells the same volume story, not a difference in individual willingness to pay.

In [ ]:
# An observation is a state/group/time/day aggregate, NOT a person or transaction.
state_group_intensity = (
    df.groupby(['State', 'Group'], as_index=False)
      .agg(Records=('Unit', 'size'),
           Average_Units_per_Record=('Unit', 'mean'),
           Average_Sales_per_Record=('Sales', 'mean'),
           Total_Sales=('Sales', 'sum'))
)
print('Average units and sales per recorded period in the lower-selling states:')
display(state_group_intensity[
    state_group_intensity['State'].isin(low_state_names)
].sort_values(['State', 'Average_Units_per_Record'], ascending=[True, False]).round(2))

**Next: compare the higher- and lower-volume records within each state.** We use each state's own 25th and 75th percentiles of `Unit` as thresholds, rather than comparing all states against Victoria's much larger numbers.

- **Low:** a record with `Unit` at or below its state's 25th-percentile cutoff.
- **High:** a record with `Unit` at or above its state's 75th-percentile cutoff.
- **Middle:** the remaining records.

Equal values at a cutoff are included, so these bands are **not necessarily exactly 25% of records**. They are units sold in aggregated time slots, **not quartiles of customer spend**.

In [ ]:
# Calculate the two quartile cutoffs within each state, then label every record.
state_volume_thresholds = (
    df.groupby('State')['Unit']
      .quantile([0.25, 0.75])
      .unstack()
      .rename(columns={0.25: 'Low_cutoff_units', 0.75: 'High_cutoff_units'})
)
records_with_bands = df.join(state_volume_thresholds, on='State').copy()
records_with_bands['Volume_Band'] = 'Middle'
records_with_bands.loc[
    records_with_bands['Unit'] <= records_with_bands['Low_cutoff_units'], 'Volume_Band'
] = 'Low'
records_with_bands.loc[
    records_with_bands['Unit'] >= records_with_bands['High_cutoff_units'], 'Volume_Band'
] = 'High'

record_volume_bands = (
    records_with_bands.groupby(['State', 'Volume_Band'], as_index=False)
      .agg(Records=('Unit', 'size'), Mean_Units=('Unit', 'mean'),
           Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
)
record_volume_bands['State_Sales_Share_pct'] = (
    100 * record_volume_bands['Total_Sales'] /
    record_volume_bands.groupby('State')['Total_Sales'].transform('sum')
)
print('Unit thresholds for the lower-selling states:')
display(state_volume_thresholds.loc[low_state_names])
print('Low / middle / high recorded periods and their sales share:')
display(record_volume_bands[
    record_volume_bands['State'].isin(low_state_names)
].sort_values(['State', 'Volume_Band']).round(1))

**Interpretation for our recommendations:** In Western Australia, records of **11 or more units** fall in the high-volume band and account for about **42.5% of recorded sales**; records of **5 or fewer units** fall in the low-volume band and account for **12.4%**. These are categories of recorded selling periods, not people or their shopping baskets.

A sensible initial strategy is to **protect product availability in busier periods**, while **testing promotions or assortment adjustments in quieter periods**. But discounts may cut profit or shift existing sales rather than generate new demand. Measure *incremental profit* against a control before scaling. Do not discontinue Women, Men, Kids or Seniors on the evidence here: all four have material sales.

**Data needed for a real high-spend-vs-low-spend customer strategy:** transaction count, basket-level quantities, actual paid prices, customer/visit identifiers (privacy-appropriate), product mix, returns, margin, inventory and store-level footfall. Without those, we cannot decide whether AAL should target fewer high-spending customers, help smaller baskets grow, or simply attract more visits.

**What do the time summaries suggest?** November has the lowest recorded monthly sales (**90,682,500**), while December has the highest (**135,330,000**). A seasonal explanation is *possible*, but one Q4 dataset does not demonstrate a recurring trend. We should avoid treating Halloween or New Year's Eve as confirmed closures.

### Section 2 verification

These checks make sure the totals reconcile across the demographic, geographic and time-based reports. The tables must all add back to the original source's recorded sales and units; none should create or discard data.

In [ ]:
for report in [group_report, state_report, weekly_report, monthly_report, quarterly_report]:
    assert report['Total_Sales'].sum() == df['Sales'].sum()
    assert report['Total_Units'].sum() == df['Unit'].sum()

assert weekly_report['Recorded_Days'].sum() == df['Date'].nunique()
assert monthly_report['Recorded_Days'].sum() == df['Date'].nunique()
assert quarterly_report['Recorded_Days'].sum() == df['Date'].nunique()

assert segment_sales.to_numpy().sum() == df['Sales'].sum()
assert (segment_share.sum(axis=1) - 100).abs().max() < 1e-8
assert record_volume_bands['Total_Sales'].sum() == df['Sales'].sum()
assert record_volume_bands['Records'].sum() == len(df)
assert (record_volume_bands.groupby('State')['State_Sales_Share_pct'].sum() - 100).abs().max() < 1e-8
print('Section 2 checks passed: all group, state, demographic segment and period totals reconcile.')

---
## 3. Data Visualization

**Rubric goal:** build a clear, accessible **sales dashboard** for the head of Sales & Marketing, covering (1) states and demographic groups, (2) groups across states, (3) peak/off-peak time of day, and (4) **daily, weekly, monthly and quarterly** charts.

We will plot the actual **recorded** `Sales` totals. The course source does not specify currency, so chart labels say **sales values**, not dollars. We keep the two completely unrecorded dates as gaps rather than treating them as zero sales. Section 3 builds and explains the plots; the final written report belongs in Section 4.

Our toolkit is **Seaborn + Matplotlib**. Seaborn is useful for labelled statistical plots and heatmaps; Matplotlib lets us control titles, axes and calendar dates.

In [ ]:
# These imports work in JupyterLab. Seaborn builds on Matplotlib.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
state_order = state_report['State'].tolist()  # Sort by total sales, largest first
customer_groups = ['Kids', 'Women', 'Men', 'Seniors']
time_order = ['Morning', 'Afternoon', 'Evening']
print('States:', state_order)
print('Demographics:', customer_groups)

### 3a. State-wise sales for each demographic group

A **heatmap** shows 7 states/territories × 4 customer groups. Each square contains a sales value in **millions**, which helps us compare stronger and weaker combinations. We use the original `Sales`, not the normalized values (normalization changes the units).

In [ ]:
state_group_sales = (
    df.groupby(['State', 'Group'])['Sales'].sum()
      .unstack('Group')
      .reindex(index=state_order, columns=customer_groups)
)
state_group_chart_data = state_group_sales / 1_000_000  # Only for axis labels

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(state_group_chart_data, annot=True, fmt='.2f', cmap='Blues',
            linewidths=.6, linecolor='white', cbar_kws={'label': 'Sales values (millions)'},
            ax=ax)
ax.set(title='Recorded sales by state and demographic | Q4 2020',
       xlabel='Demographic group', ylabel='State / territory')
fig.tight_layout()
plt.show()

**How to read it:** darker cells indicate higher recorded sales, but raw sales alone do not show which range is more profitable. The state and group table in Section 2 supplies the exact values behind this chart.

### 3b. Group-wise sales across Australian states

A **grouped bar chart** reverses the question: for each demographic, which state contributes the most sales? Multiple bars per group give a direct comparison; we label every state's series in the legend.

In [ ]:
group_state_chart_data = state_group_chart_data.T

fig, ax = plt.subplots(figsize=(12, 5.5))
group_state_chart_data.plot(kind='bar', ax=ax, width=.82)
ax.set(title='Each demographic group across states | Q4 2020',
       xlabel='Customer group', ylabel='Recorded sales values (millions)')
ax.set_xticklabels(group_state_chart_data.index, rotation=0)
ax.legend(title='State / territory', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()

### 3c. Peak and off-peak sales by time of day

The CSV records **Morning, Afternoon, and Evening** rather than clock times. We can compare these three periods, but cannot calculate exact opening hours or hourly peaks. On the right, a second heatmap shows each state's **percentage** of sales by time period. Percentages make the daypart patterns comparable across differently sized states.

Since all three periods have equally many rows in the source, comparing total recorded sales across them is reasonable. This does **not** measure customer-level spending.

In [ ]:
time_of_day_totals = df.groupby('Time')['Sales'].sum().reindex(time_order)
peak_period = time_of_day_totals.idxmax()
off_peak_period = time_of_day_totals.idxmin()

state_time_sales = (
    df.groupby(['State', 'Time'])['Sales'].sum()
      .unstack('Time').reindex(index=state_order, columns=time_order)
)
state_time_sales_share = state_time_sales.div(state_time_sales.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), gridspec_kw={'width_ratios': [1, 1.45]})
axes[0].bar(time_order, time_of_day_totals / 1_000_000, color=['#377eb8', '#4daf4a', '#984ea3'])
axes[0].set(title='National sales by daypart', xlabel='Time of day',
            ylabel='Recorded sales values (millions)')
for x, total in enumerate(time_of_day_totals / 1_000_000):
    axes[0].text(x, total + 1.2, f'{total:.1f}', ha='center', fontsize=10)
axes[0].set_ylim(0, time_of_day_totals.max() / 1_000_000 * 1.16)
sns.heatmap(state_time_sales_share, annot=True, fmt='.1f', cmap='Blues',
            cbar_kws={'label': 'Share of state daypart sales (%)'}, ax=axes[1])
axes[1].set(title='Daypart mix within each state (%)',
            xlabel='Time of day', ylabel='State / territory')
fig.tight_layout()
plt.show()
print('Peak recorded sales period:', peak_period, f'({time_of_day_totals.max():,.0f})')
print('Off-peak recorded sales period:', off_peak_period, f'({time_of_day_totals.min():,.0f})')

**Business interpretation:** a period with higher recorded sales could merit careful stock and staffing checks. Lower-sales periods might be suitable for controlled marketing tests. Differences are modest; we cannot establish why they exist or whether discounts would improve profit.

### 3d. Daily, weekly, monthly and quarterly sales charts

The rubric explicitly asks for **all four calendar resolutions**. We plot each separately and explain its caveats. Because **31 October and 31 December 2020 have no records**, missing daily values must appear as **gaps, not zero**.

**Daily:** The line plots total sales for each recorded date. The dashed lines flag dates for which we have no observations. A blank point is not evidence that the stores were closed.

In [ ]:
daily_chart_data = df.groupby('Date')['Sales'].sum().reindex(quarter_dates)

fig, ax = plt.subplots(figsize=(12, 4.3))
ax.plot(daily_chart_data.index, daily_chart_data / 1_000_000,
        color='#23638d', lw=1.8, marker='o', markersize=2.5)
for i, date in enumerate(missing_dates):
    ax.axvline(date, color='#c35145', lw=1.4, ls='--', alpha=.8,
               label='Unrecorded date' if i == 0 else None)
ax.set(xlim=(quarter_dates.min(), quarter_dates.max()),
       title='Daily recorded sales | Q4 2020', xlabel='Date',
       ylabel='Recorded sales values (millions)')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()
print('Unrecorded dates (not zero sales):', [d.strftime('%d %b %Y') for d in missing_dates])

**Weekly:** Each bar includes the total sales from the observed days in that Monday–Sunday week. Weeks containing fewer than **7 recorded Q4 days** are labelled with their day count. Their totals are **not directly comparable** with complete seven-day weeks.

In [ ]:
weekly_chart_data = weekly_report.copy()
weekly_chart_data['Week_Start'] = weekly_chart_data['Week'].apply(lambda p: p.start_time)
weekly_labels = weekly_chart_data['Week_Start'].dt.strftime('%d %b')

fig, ax = plt.subplots(figsize=(12, 4.7))
bars = ax.bar(weekly_labels, weekly_chart_data['Total_Sales'] / 1_000_000, color='#377eb8')
for bar, n_days in zip(bars, weekly_chart_data['Recorded_Days']):
    if n_days < 7:
        ax.annotate(f'{n_days} days',
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=9, color='#9a3931')
ax.set(title='Weekly sales totals | Weeks beginning Monday',
       xlabel='Week starting', ylabel='Recorded sales values (millions)')
ax.tick_params(axis='x', rotation=45)
ax.set_ylim(0, max(weekly_chart_data['Total_Sales']) / 1_000_000 * 1.23)
fig.tight_layout()
plt.show()

**Monthly:** October, November and December each have **30 recorded days**, so their totals are comparable on *observed days*, although they do not prove a recurring seasonal pattern.

In [ ]:
monthly_chart_data = monthly_report.copy()
monthly_labels = monthly_chart_data['Month'].dt.strftime('%b %Y')

fig, ax = plt.subplots(figsize=(8.5, 4.3))
bars = ax.bar(monthly_labels, monthly_chart_data['Total_Sales'] / 1_000_000,
              color=['#4c78a8', '#72b7b2', '#e59c4c'])
for bar, val in zip(bars, monthly_chart_data['Total_Sales'] / 1_000_000):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 2, f'{val:.1f}', ha='center')
ax.set(title='Monthly recorded sales | October–December 2020',
       xlabel='Month', ylabel='Recorded sales values (millions)')
ax.set_ylim(0, monthly_chart_data['Total_Sales'].max() / 1_000_000 * 1.2)
fig.tight_layout()
plt.show()

**Quarterly:** The file contains **only Q4 2020**, so a quarterly chart has **one bar**. This reports the quarter's total; it does **not** establish quarter-on-quarter growth.

In [ ]:
quarterly_chart_data = quarterly_report.copy().reset_index(drop=True)

fig, ax = plt.subplots(figsize=(6.5, 4.3))
sales_millions = quarterly_chart_data['Total_Sales'] / 1_000_000
ax.bar(quarterly_chart_data['Quarter'].astype(str), sales_millions, color='#377eb8', width=.48)
for x, value in enumerate(sales_millions):
    ax.text(x, value + 7, f'{value:.2f}M', ha='center', fontsize=11, fontweight='bold')
ax.set(title='Q4 2020 total recorded sales | One quarter only',
       xlabel='Quarter', ylabel='Recorded sales values (millions)')
ax.set_ylim(0, sales_millions.max() * 1.23)
fig.tight_layout()
plt.show()

### 3e. Statistical plots: box plot and distribution

The assignment's final reporting guidance calls for a **box plot** for descriptive statistics and a **Seaborn distribution** plot. We create them here so Section 4 can concentrate on the written explanation and recommendations.

A **box plot** shows the median (line), middle 50% (box), whiskers and possible outliers. The figure uses `Unit` because sales values are simply units multiplied by 2,500.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.boxplot(data=df, x='Group', y='Unit', order=customer_groups,
            color='#7da6c8', ax=ax)
ax.set(title='Distribution of units per recorded row by demographic',
       xlabel='Customer group', ylabel='Units per recorded row')
fig.tight_layout()
plt.show()

A **histogram** counts how many recorded rows fall into each unit-volume interval. We use Seaborn's current `histplot()` function instead of the deprecated `distplot()`. This is a distribution of **aggregated recorded periods**, not individual purchases.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.histplot(data=df, x='Unit', bins=24, color='#377eb8', ax=ax)
ax.set(title='How frequently do different unit volumes occur?',
       xlabel='Units in a recorded row', ylabel='Number of recorded rows')
fig.tight_layout()
plt.show()

### 3f. Executive dashboard: four plots on one page

The charts above teach each method individually. This **static dashboard** combines state-by-group sales, state ranking, time-of-day and monthly totals into one view for the head of Sales & Marketing. It is designed for straightforward JupyterLab viewing, not live filtering.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.heatmap(state_group_chart_data, annot=True, fmt='.1f', cmap='Blues',
            linewidths=.4, cbar=False, ax=axes[0, 0])
axes[0, 0].set(title='By state and demographic (millions)',
               xlabel='Customer group', ylabel='State / territory')

state_summary = state_report.sort_values('Total_Sales')
axes[0, 1].barh(state_summary['State'], state_summary['Total_Sales'] / 1_000_000,
                color='#377eb8')
axes[0, 1].set(title='State/territory totals', xlabel='Sales values (millions)',
               ylabel='State / territory')

axes[1, 0].bar(time_order, time_of_day_totals / 1_000_000, color='#2c866f')
axes[1, 0].set(title='Time of day (national)', xlabel='Daypart',
               ylabel='Sales values (millions)')

axes[1, 1].bar(monthly_labels, monthly_chart_data['Total_Sales'] / 1_000_000,
                color='#b88645')
axes[1, 1].set(title='Monthly totals', xlabel='Month',
               ylabel='Sales values (millions)')

fig.suptitle('AAL | Q4 2020 recorded sales dashboard', fontsize=17, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, .96))
plt.show()

**Why Seaborn and Matplotlib?** The course prefers Seaborn, and it works naturally with Pandas. It produces readable heatmaps, box plots and distributions with minimal code. Matplotlib supplies flexible date formatting, axis labels, figure layout and the four-panel dashboard. Our charts have clear titles, axes, measurement units, legends and annotations; we do not rely on colour alone for every important comparison.

**Interpretation limits:** sales value is **not profit**; the data has no customer-level basket spending, margins, store counts or precise sale times. The apparent daypart differences are small, and the two completely unrecorded dates are not proof of closures. Targeted marketing campaigns should be tested before removing demographic ranges.

**Section 3 verification:** All chart totals must reconcile to the original Q4 source. The two absent dates must remain `NaN` on the daily timeline, not zero.

In [ ]:
expected_total_sales = df['Sales'].sum()
assert state_group_sales.to_numpy().sum() == expected_total_sales
assert abs(group_state_chart_data.to_numpy().sum() - expected_total_sales / 1_000_000) < 1e-9
assert time_of_day_totals.sum() == expected_total_sales
assert (state_time_sales.sum(axis=1) == state_group_sales.sum(axis=1)).all()
assert len(daily_chart_data) == len(quarter_dates) == 92
assert daily_chart_data.isna().sum() == 2
assert pd.isna(daily_chart_data.loc[pd.Timestamp('2020-10-31')])
assert pd.isna(daily_chart_data.loc[pd.Timestamp('2020-12-31')])
assert daily_chart_data.sum() == expected_total_sales
for report in (weekly_chart_data, monthly_chart_data, quarterly_chart_data):
    assert report['Total_Sales'].sum() == expected_total_sales
print('Section 3 checks passed: all chart totals reconcile and missing dates remain gaps.')

---
## 4. Report Generation

**Final report for the Head of Sales & Marketing, AAL**  
**Reporting period:** 1 October to 31 December 2020 (90 days recorded)  
**Scope:** Seven Australian states/territories, four demographic groups, three dayparts.

The earlier sections document the complete, reproducible process: **data wrangling (1)**, **statistical analysis (2)** and **data visualization (3)**. This final section interprets the evidence and makes proportionate recommendations. The figures below use *recorded Sales values*, not an assumed currency, and **do not represent net profit**.

### 4a. Executive summary

**Main conclusion:** AAL's recorded Q4 sales vary much more by location and month than by demographic group. November is consistently weaker than October across states, while December is the strongest recorded month. This makes a **small, measurable November sales-programme pilot** a stronger first experiment than removing demographic lines.

| Executive measure | Result |
|---|---:|
| Recorded Q4 sales value | **340,302,500** |
| Units represented | **136,121** |
| Records | **7,560** |
| State with highest recorded sales | **Victoria (VIC): 105,565,000** |
| State with lowest recorded sales | **Western Australia (WA): 22,152,500** |
| Highest demographic total | **Men: 85,750,000** |
| Lowest demographic total | **Seniors: 84,037,500** |
| Highest and lowest recorded months | **December: 135,330,000; November: 90,682,500** |

**Decision request:** Approve a controlled November campaign experiment, regional assortment research, and collection of cost/transaction data before any wider investment or demographic withdrawal.

### 4b. Methodology: what the data can and cannot tell us

The source file is the course-provided `AusApparalSales4thQrt2020.csv` (inside the original ZIP); the project brief is `sales_analysis_problem_statement.docx`. The unit of observation is one **date x state x demographic x time-of-day** combination. An observation is **not** a customer, receipt or individual purchase.

- **Data checks (Section 1):** inspected types, `isna()` / `notna()`, duplicate keys, negative and malformed values; trimmed category whitespace; parsed dates; kept the original data untouched.
- **Missing dates:** 31 October and 31 December have no records. No null replacement can recover *entire missing dates*. They were **not imputed or treated as zero sales**.
- **Normalization:** Min-Max created separate 0-to-1 columns for `Unit` and `Sales`, without overwriting the original values. Original sales are used for monetary-scale aggregation and charts.
- **GroupBy and statistics (Section 2):** calculated totals and mean, median, mode and standard deviation; used Monday-Sunday weeks, months, and the only available quarter.
- **Charts (Section 3):** Seaborn for heatmaps, histograms and box plots; Matplotlib for time axes, grouped plots and the executive dashboard. The source does not establish customer spending or exact hourly timing.

**Special data characteristic:** Every row has `Sales = 2,500 x Unit`. The same fixed ratio holds across groups, places and days, so the two columns are proportional. The ratio is an observed arithmetic property, **not evidence that every item was actually sold at a constant retail price**. We cannot infer basket spend, margin, customers or price sensitivity from these aggregates.

### 4c. Results and interpretation

**Geography:** Victoria accounts for the greatest recorded sales, while Western Australia, Northern Territory and Tasmania have substantially smaller totals. All states have the same number of source records, but this does not mean equal store counts, footfall or market sizes. Low total revenue is **not** proof of weak performance per store.

**Demographics:** Men lead nationally and Seniors have the lowest total, but their difference is small against the whole quarter. Even in the four lower-selling states, every customer group contributes approximately one quarter of recorded sales. There is **no support here for eliminating any demographic range**.

**Time of day:** Morning is the highest recorded daypart at 114,207,500; Evening the lowest at 112,087,500. The difference is modest, so daypart-targeted offers are a hypothesis worth testing, not an established solution.

**Seasonality:** This dataset covers **only Q4 2020**, so a month-to-month pattern is not proof that November is always weak or that December routinely peaks.

In [ ]:
# Executive reporting table: recorded sales, not profit.
location_summary = state_report.copy()
location_summary['Share_of_Q4_pct'] = (
    100 * location_summary['Total_Sales'] / df['Sales'].sum()
)
location_summary['Average_Sales_per_Recorded_Row'] = (
    location_summary['Total_Sales'] / location_summary['Records']
)
print('Regional performance table (unadjusted for store count and local demand):')
display(location_summary.round({'Share_of_Q4_pct': 2,
                                'Average_Sales_per_Recorded_Row': 2}))

print('Lowest-selling states and their highest / lowest demographic groups:')
display(low_state_demographics.round(2))

**Month-by-month comparison:** October, November and December each have exactly 30 recorded dates. November is **20.7% below October** in recorded sales; every one of the seven states also declines from October to November. We should investigate whether AAL was already discounting in 2020, and avoid projecting a single quarter as an annual seasonal rule.

In [ ]:
# Build a monthly sales table and check November across states.
monthly_final = monthly_report.copy()
monthly_final['Month_Name'] = monthly_final['Month'].astype(str)
monthly_final['Sales_vs_Oct_pct'] = (
    100 * (monthly_final['Total_Sales'] / monthly_final.loc[0, 'Total_Sales'] - 1)
)
print('Monthly recorded sales (each with 30 observed days):')
display(monthly_final[['Month_Name','Recorded_Days','Total_Units',
                       'Total_Sales','Sales_vs_Oct_pct']].round(2))

state_month_sales = (
    df.assign(Month=df['Date'].dt.month)
      .groupby(['State', 'Month'])['Sales'].sum()
      .unstack('Month').rename(columns={10:'October', 11:'November', 12:'December'})
)
state_month_sales['Nov_vs_Oct_pct'] = (
    100 * (state_month_sales['November'] / state_month_sales['October'] - 1)
)
print('All states: November relative to October:')
display(state_month_sales.sort_values('Nov_vs_Oct_pct').round(2))

# For completeness, check the same relationship in every state-by-group pair.
state_group_month = (
    df.assign(Month=df['Date'].dt.month)
      .groupby(['State', 'Group', 'Month'])['Sales'].sum()
      .unstack('Month')
)
num_state_group_declines = int((state_group_month[11] < state_group_month[10]).sum())
print('State / demographic combinations with lower November than October:',
      num_state_group_declines, 'of', len(state_group_month))

**Visualization of the November opportunity:** The grouped chart below shows whether November's smaller total is limited to one market or extends across regions. A difference in recorded sales is not itself the causal effect of a marketing campaign.

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.5))
state_month_sales[['October','November','December']].plot(
    kind='bar', ax=ax, width=0.78)
ax.set(title='Recorded sales by state and month | Q4 2020',
       xlabel='State / territory', ylabel='Recorded sales values (millions)')
# Rescale tick labels rather than scaling the underlying source values.
from matplotlib.ticker import FuncFormatter
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x / 1_000_000:,.0f}'))
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Month')
fig.tight_layout()
plt.show()

**Additional teaching-data checks:** The two unrecorded dates are exactly the 31st days of October and December. Every month has 30 recorded days and there are 84 records for each observed day (7 states x 4 groups x 3 dayparts). The dataset also has sales recorded on **25 December 2020**. This is more consistent with a deliberate 30-day-per-month coverage rule or export truncation than with confirmed individual holiday closures, although the cause **cannot be established** from the CSV. The brief is a course scenario, so we must avoid generalising its patterns to real retail operations.

In [ ]:
# Check dates and the often-mentioned Black Friday hypothesis.
observed_by_month = df.groupby(df['Date'].dt.month)['Date'].nunique()
christmas_day_present = pd.Timestamp('2020-12-25') in set(df['Date'])
nov_27 = df.loc[df['Date'] == pd.Timestamp('2020-11-27'), 'Sales'].sum()
nov_average_day = monthly_report.loc[
    monthly_report['Month'].astype(str) == '2020-11', 'Total_Sales'
].iloc[0] / 30
print('Recorded dates in Oct, Nov, Dec:', observed_by_month.to_dict())
print('25 December recorded:', christmas_day_present)
print('27 November 2020 recorded sales:', f'{nov_27:,.0f}')
print('November average daily recorded sales:', f'{nov_average_day:,.0f}')
print('27 November versus November daily average:',
      f'{(nov_27 / nov_average_day - 1) * 100:.1f}%')

### 4d. Recommended sales and marketing programme

**Priority 1: Trial a November campaign rather than launching an automatic nationwide sale.** November is the weakest recorded month in all seven states, and Black Friday provides a possible campaign window. AAL should first establish existing promotions, inventory and margin; then test a modest offer against comparable control stores or audiences. Measure the **incremental contribution profit** after discounts, returns, fulfilment and campaign costs. Also measure *combined November-December outcomes* to detect sales brought forward from December. In this dataset, Black Friday (27 November 2020) is **not a visible spike** above the average November day; historical promotional success has not been demonstrated.

**Priority 2: Research and test targeted offers in lower-selling states.** Start with local customer research and inventory review before promotion design. Possible *hypotheses*, not proven targets, include Women's products in **Western Australia** and Seniors' products in **Northern Territory**, where these groups have the lowest recorded sales totals within their respective states. Protect stock for stronger categories at the same time. The difference between customer groups is too small to justify withdrawing any entire demographic range.

**Priority 3: Review trading periods and stock availability.** Morning recorded sales slightly exceed Evening sales. Test limited evening offers or assortment availability only where stores have sufficient capacity and healthy margins, rather than diverting national marketing spend solely on this modest difference.

**Priority 4: Use normalized and per-store comparisons in future analyses.** Revenue per store, gross margin, stock turns, regional population and footfall would give far more useful location comparisons than simple totals. Avoid treating the number of dataset rows as a proxy for store count.

| Pilot | Initial target | What we would measure | Decision rule |
|---|---|---|---|
| November promotion | Matched stores/audiences in selected states | Incremental contribution profit, units, returns, November+December | Expand only if profit uplift is demonstrated |
| Regional assortment | WA Women, NT Seniors as research starting points | Margin, stock turn, conversion, product availability | Keep or revise ranges from actual performance |
| Off-peak experiment | Selected Evening trading periods | Profit per open hour, net lift, staff and campaign costs | Continue only if incremental economics are positive |

The suggested demographic experiments are **inclusive product and marketing tests**, not decisions to exclude particular customers.

**Why not simply offer a 15% discount everywhere?** A discount can sell more units without improving revenue or profitability. The illustration below is deliberately hypothetical; the supplied CSV does not contain the margin or price data required to predict a true promotion response.

In [ ]:
# Illustrative arithmetic only: not estimated from the AAL source.
discount_fraction = 0.15
hypothetical_unit_increase = 0.10
revenue_ratio = (1 - discount_fraction) * (1 + hypothetical_unit_increase)
break_even_unit_increase_for_revenue = 1 / (1 - discount_fraction) - 1
print('If price falls 15% and volume rises 10%:')
print(f'  New revenue as a share of original revenue: {revenue_ratio:.1%}')
print(f'  Change in revenue: {(revenue_ratio - 1):.1%}')
print('  Unit-volume increase needed just to preserve revenue:',
      f'{break_even_unit_increase_for_revenue:.1%}', 'before considering costs')

### 4e. Pilot evaluation and data required

**Proposed experiment:** Choose comparable stores or audiences within each selected region and, where practical, randomly assign promotion and control. Run the same dates with consistent attribution. Compare against a pre-pilot baseline. Record promotion eligibility, products and spend, and check whether increases persist when the campaign ends.

**Primary KPI:** change in **contribution profit** compared with the control group, *after* discounts, cost of goods, returns, fulfilment and campaign spend. Use actual cost data, not the `Sales` field alone.

**Secondary KPIs:** incremental units, revenue, transaction count, average basket value, conversion rate, repeat purchases, stockouts, return rate, performance by group/state/daypart, and combined November+December results. Use privacy-conscious aggregation for individual customer information.

**Decision rule:** Do not scale merely because revenue increases. Scale only where the test supports a meaningful positive contribution margin without unacceptable stockouts, customer harm or December cannibalisation. If the dataset is kept as a teaching simulation, report these as a proposed evaluation design, not an experiment already conducted.

**Missing operational data to request:** unit-level transaction/receipt data, actual prices and discounts, profit margins, costs and returns, inventory and replenishment, store and region attributes, customer counts, footfall, campaign history, and at least a full year (preferably several years) of comparable sales dates.

### 4f. Limitations, uncertainties and source references

1. **Teaching dataset:** the brief is a course-end business scenario; patterns should not be asserted as real-world retail facts. The CSV may be generated or standardised, but its provenance cannot be proven from the file alone.
2. **Calendar:** 31 October and 31 December 2020 are missing; the cause is unconfirmed. Holiday closures should **not** be reported as observed facts, nor should the days be assigned zero revenue.
3. **Coverage:** just one quarter, only seven state/territory categories, no Australian Capital Territory. There are exactly 30 observed days in each month.
4. **Unit of observation:** each row aggregates day, state, demographic and daypart, not a shopping basket or customer. No valid conclusions about individual spending, price sensitivity or customer lifetime value can be drawn.
5. **Pricing and profit:** the mechanical `Sales = Unit x 2,500` relationship and unspecified currency prevent inference of transaction prices, margins, return rates or net profits. Higher sales alone do not establish higher returns on marketing investment.
6. **Comparability:** states may differ in store footprint, customer base and population, none of which is measured. Low totals do not imply low per-store productivity.
7. **Causality:** November's recorded dip and the Black Friday observation do not demonstrate why sales changed, nor whether future promotional activity would be effective.

**Report sources:** the course's original `sales_analysis_problem_statement.docx` and `AusApparalSales4thQrt2020.csv` inside `1739523823_ausapparalsales4thqrt2020.zip`. No public market benchmarks or external financial data were used.

### Closing recommendation

**Preserve all four demographic ranges while validating profitability.** AAL should investigate regional differences, then run a **controlled November promotion trial** with store-level and product-level economics. Use the outcome to decide where to expand marketing, rather than mistaking this tidy training dataset for proof of customer behaviour.

### 4g. Rubric coverage checklist

| Course requirement | Evidence in this notebook |
|---|---|
| 1a-b. Missing/incorrect data and handling | Section 1: `isna()`, `notna()`, key checks, dates and rationale |
| 1c. Normalization | Section 1: Min-Max on Unit and Sales, preserving raw fields |
| 1d. GroupBy use | Section 1: state and group aggregation, grouping rather than merging |
| 2a-c. Descriptive statistics and high/low groups | Section 2: mean, median, mode, SD, demographic and state rankings |
| 2d. Weekly, monthly and quarterly reports | Section 2: dated grouping with coverage counts |
| 3a-b. Dashboard and accessible charts | Section 3: state/group, daypart, daily/weekly/monthly/quarterly, box plot and distribution |
| 3c. Choice of graphics package | Section 3: Seaborn and Matplotlib rationale |
| 4a-c. JupyterLab report, Markdown, plots, recommendations | Sections 1-4; final executive summary, findings, strategy and limitations |

**Note:** The brief repeats the demographic-group question under 2b and 2c. Section 2 also ranks states, addressing the business request to identify high- and low-revenue regions.

In [ ]:
# Final integrity checks. These are checks on the course dataset, not proof of promotion impact.
assert len(df) == 7560
assert int(df['Unit'].sum()) == 136121
assert int(df['Sales'].sum()) == 340302500
assert monthly_final['Recorded_Days'].tolist() == [30, 30, 30]
assert int(monthly_final.loc[1, 'Total_Sales']) == 90682500
assert int(df.groupby('Date')['Sales'].sum().loc[pd.Timestamp('2020-11-27')]) == 2982500
assert (state_month_sales['Nov_vs_Oct_pct'] < 0).all()
assert num_state_group_declines == 28
assert int(df.groupby('Date').size().min()) == 84
assert int(df.groupby('Date').size().max()) == 84
assert df.groupby('Date').size().shape[0] == 90
assert observed_by_month.to_dict() == {10:30,11:30,12:30}
assert len(missing_dates) == 2
assert set(missing_dates) == {pd.Timestamp('2020-10-31'), pd.Timestamp('2020-12-31')}
assert abs(revenue_ratio - 0.935) < 1e-12
assert state_group_month[11].lt(state_group_month[10]).all()
print('AAL Final Report: checks passed (coverage, totals, findings and scenario arithmetic).')